# Main paper figures

Notebook for reproducing the main figures of the paper "Testing the Limits of Truth Directions in LLMs". 

*Refactored/cleaned up using Claude Code (Opus 4.8)

In [ ]:
# Configs
import pickle, csv
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "DejaVu Sans"})

MODEL_NAME  = "Llama-3.1-8B-Instruct"
NUM_LAYERS  = {"Llama-3.1-8B-Instruct": 32, "Llama-3.2-3B-Instruct": 28,
               "gemma-2-9b-it": 42, "gemma-2-2b-it": 26}[MODEL_NAME]
LAYERS      = list(range(NUM_LAYERS))

PROBES_DIR = Path("/probes")
ACTS_DIR   = Path("/activations")

INSTRUCTIONS = {
    "no-prompt":   "no-prompt",
    "ask-correct": "ask-correct",
    "ask-tf":      "ask-tf",
    "ask-able":    "ask-able",
}

ARITH_DS = ["arith_1op", "arith_2ops", "arith_3ops"]
FACT_DS  = ["cities", "neg_cities", "cities_conj",
            "cities_same_country_quant", "cities_exact_k", "cities_exact_k1_k2"]
ALL_DS   = ARITH_DS + FACT_DS

LABEL = {
    "arith_1op": "A1", "arith_2ops": "A2", "arith_3ops": "A3",
    "cities": "F0", "neg_cities": "F1", "cities_conj": "F2",
    "cities_same_country_quant": "F3", "cities_exact_k": "F4",
    "cities_exact_k1_k2": "F5",
}
DS_LABELS = [LABEL[ds] for ds in ALL_DS]
MARKERS   = ["o", "s", "^", "D", "v", "<", ">", "p", "*"]

print(f"Model: {MODEL_NAME}  ({NUM_LAYERS} layers)")

In [ ]:
# Some shared helper functions for loading probes, activations, and metrics.

def run_tag(ds, instruction):
    return f"{ds}_{instruction}_{MODEL_NAME}"

def read_pickle_stream(path):
    out = []
    if not path or not path.exists():
        return out
    with open(path, "rb") as f:
        while True:
            try:
                batch = pickle.load(f)
            except EOFError:
                break
            out.extend(batch if isinstance(batch, (list, tuple)) else [batch])
    return out

def to_layer_map(d):
    m = {}
    for k, v in d.items():
        if isinstance(k, int):
            m[k] = np.asarray(v).ravel()
        else:
            try:
                m[int(str(k).split(".")[1])] = np.asarray(v).ravel()
            except Exception:
                continue
    return m

def load_probe(ds, instruction):
    p = PROBES_DIR / f"{run_tag(ds, instruction)}_probes.pkl"
    if not p.exists():
        raise FileNotFoundError(f"No probe: {p}")
    with open(p, "rb") as f:
        obj = pickle.load(f)
    return to_layer_map(obj["directions"]), to_layer_map(obj.get("means", {}))

def load_metrics(ds, instruction):
    p = PROBES_DIR / f"{run_tag(ds, instruction)}_metrics.csv"
    if not p.exists():
        return {}
    out = {}
    with open(p, newline="") as f:
        for r in csv.DictReader(f):
            try:
                L = int(r.get("layer_idx", r.get("layer", "")))
                au = float(r.get("auroc", ""))
            except Exception:
                continue
            if not np.isnan(au):
                out[L] = au
    return out

def last_token_vec(rec, L):
    A = rec["resid_activations"].get(f"blocks.{L}.hook_resid_post")
    if A is None:
        return None
    A = np.asarray(A)
    return A[-1] if A.ndim > 1 else A

def score_vec(w, mu, v):
    if v is None or v.shape != w.shape:
        return None
    if mu is not None and mu.shape == w.shape:
        v = v - mu
    return float(np.dot(w, v))

def load_activations(ds, instruction):
    tag = run_tag(ds, instruction)
    pos = read_pickle_stream(ACTS_DIR / f"{tag}_last_run_correct.pkl")
    neg = read_pickle_stream(ACTS_DIR / f"{tag}_last_run_incorrect.pkl")
    return pos, neg

def eval_generalization(train_ds, instruction, tgt_ds, layers):
    try:
        w_by_L, mu_by_L = load_probe(train_ds, instruction)
    except FileNotFoundError:
        return {L: np.nan for L in layers}
    pos, neg = load_activations(tgt_ds, instruction)
    out = {}
    for L in layers:
        w, mu = w_by_L.get(L), mu_by_L.get(L)
        if w is None:
            out[L] = np.nan
            continue
        scores, labels = [], []
        for rec in pos:
            s = score_vec(w, mu, last_token_vec(rec, L))
            if s is not None:
                scores.append(s) 
                labels.append(1)
        for rec in neg:
            s = score_vec(w, mu, last_token_vec(rec, L))
            if s is not None:
                scores.append(s) 
                labels.append(0)
        out[L] = roc_auc_score(labels, scores) if scores else np.nan
    return out

def auroc_array(ds, instruction):
    m = load_metrics(ds, instruction)
    return np.array([m.get(L, np.nan) for L in LAYERS])

print("Helpers loaded.")

---
## Figure 1: Layer dependence of truth directions

In [ ]:
# Figure 1a: In-domain test AUROC per layer
INSTR = "no-prompt"

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
for i, ds in enumerate(ALL_DS):
    ax.plot(LAYERS, auroc_array(ds, INSTR), lw=3, marker=MARKERS[i], ms=5, label=LABEL[ds])
ax.set(xlabel="Layer", ylabel="AUROC", ylim=(0.45, 1.05))
ax.set_yticks(np.arange(0.5, 1.05, 0.1))
ax.tick_params(labelsize=14); ax.xaxis.label.set_size(22); ax.yaxis.label.set_size(22)
ax.legend(loc="lower right", ncol=3, fontsize=14)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"fig1a_indomain_auroc_{MODEL_NAME}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 1b: F0-probe cross-task generalization AUROC per layer
F0_DS = "cities"

gen = {}
for ds in ALL_DS:
    if ds == F0_DS:
        gen[ds] = auroc_array(F0_DS, INSTR)
    else:
        auc = eval_generalization(F0_DS, INSTR, ds, LAYERS)
        gen[ds] = np.array([auc.get(L, np.nan) for L in LAYERS])

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
for i, ds in enumerate(ALL_DS):
    ls = "--" if ds == F0_DS else "-"
    ax.plot(LAYERS, gen[ds], lw=3, ls=ls, marker=MARKERS[i], ms=5, label=LABEL[ds])
ax.set(xlabel="Layer", ylabel="AUROC", ylim=(-0.05, 1.05))
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.tick_params(labelsize=14); ax.xaxis.label.set_size(22); ax.yaxis.label.set_size(22)
ax.legend(loc="lower right", ncol=3, fontsize=14)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(f"fig1b_f0_generalization_{MODEL_NAME}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 1c: Probe evolution heatmaps (cosine similarity across layer pairs)

def cosine_matrix(ds, instruction):
    W = load_probe(ds, instruction)[0]
    M = np.full((NUM_LAYERS, NUM_LAYERS), np.nan)
    for i in range(NUM_LAYERS):
        wi = W.get(i)
        if wi is None: continue
        for j in range(NUM_LAYERS):
            wj = W.get(j)
            if wj is None: continue
            M[i, j] = np.dot(wi, wj) / (np.linalg.norm(wi) * np.linalg.norm(wj))
    return M

cos_matrices = {ds: cosine_matrix(ds, INSTR) for ds in ALL_DS}

with sns.axes_style("white"):
    fig, axes = plt.subplots(3, 3, figsize=(12, 12), dpi=300, sharex=True, sharey=True,
                             gridspec_kw=dict(hspace=0.25, wspace=0.15))
    ticks = list(range(0, NUM_LAYERS, 4))
    for idx, ds in enumerate(ALL_DS):
        ax = axes[idx // 3, idx % 3]
        im = ax.imshow(cos_matrices[ds], cmap="coolwarm", aspect="equal", origin="upper")
        ax.set_title(LABEL[ds], fontsize=22, pad=8)
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)
        ax.tick_params(labelsize=14)
        if idx // 3 == 2 and idx % 3 == 1: ax.set_xlabel("Layer $j$", fontsize=22)
        if idx % 3 == 0 and idx // 3 == 1: ax.set_ylabel("Layer $i$", fontsize=22)

    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02, aspect=30)
    cbar.set_label(r"$\cos(w_i, w_j)$", fontsize=14, rotation=270, labelpad=20)
    cbar.ax.tick_params(labelsize=14)
    fig.savefig(f"fig1c_probe_evolution_{MODEL_NAME}.pdf", bbox_inches="tight")
    plt.show()

---
## Figure 2: Layer-wise probe cosine similarity (no-prompt vs ask-correct)

In [ ]:
# Figure 2: cosine similarity hetamap between the no-prompt and ask-correct truth directions,
# per task and layer.

import pandas as pd

def cosine_per_layer(ds, instr_a, instr_b):
    Wa = load_probe(ds, instr_a)[0]
    Wb = load_probe(ds, instr_b)[0]
    out = np.full(NUM_LAYERS, np.nan)
    for L in LAYERS:
        a, b = Wa.get(L), Wb.get(L)
        if a is None or b is None or a.shape != b.shape:
            continue
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        if na == 0 or nb == 0:
            continue
        out[L] = float(np.dot(a, b) / (na * nb))
    return out

cos_by_ds  = {ds: cosine_per_layer(ds, "no-prompt", "ask-correct") for ds in ALL_DS}
cos_matrix = np.vstack([cos_by_ds[ds] for ds in ALL_DS])
df_cos = pd.DataFrame(cos_matrix,
                      index=[LABEL[ds] for ds in ALL_DS],
                      columns=[str(L) for L in LAYERS])

fig, ax = plt.subplots(figsize=(6, 3.5), dpi=300)
sns.heatmap(df_cos, ax=ax, vmin=-1.0, vmax=1.0, center=0.0, cmap="RdBu_r",
            cbar_kws={"shrink": 0.85, "aspect": 25}, xticklabels=4, yticklabels=1)
cbar = ax.collections[0].colorbar
cbar.set_label(
    r"$\cos(\mathbf{w}^{\mathrm{no\text{-}prompt}}_L,\ \mathbf{w}^{\mathrm{ask\text{-}correct}}_L)$",
    rotation=-90, labelpad=22, fontsize=13, y=0.45, va="center")
cbar.ax.tick_params(labelsize=11)
ax.set_xlabel("Layer", fontsize=20, labelpad=6)
ax.set_ylabel("")
ax.tick_params(axis="x", labelsize=13, rotation=0)
ax.tick_params(axis="y", labelsize=15, rotation=0)
ax.grid(False)
fig.tight_layout()
fig.savefig(f"fig2_cosine_no-prompt_vs_ask-correct_{MODEL_NAME}.pdf", bbox_inches="tight")
plt.show()

---
## Figure 3: Cross-template generalization (no-prompt probes on ask-correct activations)

In [ ]:
# Figure 3: probe trained on no-prompt, evaluated on ask-correct activations
colors = plt.get_cmap("tab10").colors
src_on_src, src_on_tgt = {}, {}
for ds in ALL_DS:
    w_by_L, mu_by_L = load_probe(ds, "no-prompt")
    metrics = load_metrics(ds, "no-prompt")
    pos_tgt, neg_tgt = load_activations(ds, "ask-correct")
    y_src, y_tgt = [], []
    for L in LAYERS:
        y_src.append(metrics.get(L, np.nan))
        w, mu = w_by_L.get(L), mu_by_L.get(L)
        if w is None:
            y_tgt.append(np.nan)
            continue
        scores, labels = [], []
        for rec in pos_tgt:
            s = score_vec(w, mu, last_token_vec(rec, L))
            if s is not None:
                scores.append(s)
                labels.append(1)
        for rec in neg_tgt:
            s = score_vec(w, mu, last_token_vec(rec, L))
            if s is not None:
                scores.append(s)
                labels.append(0)
        y_tgt.append(roc_auc_score(labels, scores) if scores else np.nan)
    src_on_src[ds] = np.array(y_src)
    src_on_tgt[ds] = np.array(y_tgt)

def plot_cross_prompt(ds_list, fname):
    fig, ax = plt.subplots(figsize=(9, 4), dpi=200)
    for i, ds in enumerate(ds_list):
        c = colors[i]
        ax.plot(LAYERS, src_on_src[ds], lw=2, color=c, alpha=0.35, marker=MARKERS[i], ms=5)
        ax.plot(LAYERS, src_on_tgt[ds], lw=3, color=c, ls="--",    marker=MARKERS[i], ms=5)
        ax.fill_between(LAYERS, src_on_src[ds], src_on_tgt[ds], color=c, alpha=0.10)
    ax.set(xlabel="Layer", ylabel="AUROC", ylim=(0.35, 1.05))
    ax.tick_params(labelsize=14); ax.xaxis.label.set_size(22); ax.yaxis.label.set_size(22)
    ax.legend([Line2D([],[],color=colors[i],lw=3) for i in range(len(ds_list))],
              [LABEL[ds] for ds in ds_list], ncol=3, fontsize=14)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(fname, bbox_inches="tight")
    plt.show()

plot_cross_prompt(ARITH_DS, f"fig3a_cross_prompt_arith_{MODEL_NAME}.pdf")
plot_cross_prompt(FACT_DS,  f"fig3b_cross_prompt_factual_{MODEL_NAME}.pdf")

---
## Figure 4: Cross-task generalization heatmaps

In [ ]:
# Figure 4: cross-task generalization heatmaps, one 9x9 panel per prompt template at a
# single layer. Cell (i, j) = AUROC of the probe trained on task i, evaluated on task j's
# activations under that prompt; the diagonal is in-domain held-out test AUROC.
FIG4_LAYER  = 25
FIG4_INSTRS = ["no-prompt", "ask-correct", "ask-tf", "ask-able"]
INSTR_DISPLAY = {"no-prompt": "No-prompt", "ask-correct": "Ask-correct",
                 "ask-tf": "Ask-T/F", "ask-able": "Ask-able"}
n_ds = len(ALL_DS)

def heatmap_at_layer(instruction, L):
    M = np.full((n_ds, n_ds), np.nan)
    metrics_cache = {ds: load_metrics(ds, instruction) for ds in ALL_DS}
    for i, train_ds in enumerate(ALL_DS):
        for j, test_ds in enumerate(ALL_DS):
            if train_ds == test_ds:
                M[i, j] = metrics_cache[train_ds].get(L, np.nan)
            else:
                M[i, j] = eval_generalization(train_ds, instruction, test_ds, [L]).get(L, np.nan)
    return M

print(f"Computing Figure 4 heatmaps at layer {FIG4_LAYER} ...")
fig4_maps = {instr: heatmap_at_layer(instr, FIG4_LAYER) for instr in FIG4_INSTRS}

fig, axes = plt.subplots(1, len(FIG4_INSTRS),
                         figsize=(3.6 * len(FIG4_INSTRS) + 1.2, 4.2),
                         dpi=300, constrained_layout=True, sharey=True)
im = None
for ax, instr in zip(axes, FIG4_INSTRS):
    M = fig4_maps[instr]
    im = ax.imshow(M, vmin=0, vmax=1, cmap="coolwarm", aspect="equal")
    ax.set_title(INSTR_DISPLAY[instr], fontsize=16)
    ax.set_xticks(range(n_ds)); ax.set_yticks(range(n_ds))
    ax.set_xticklabels(DS_LABELS, ha="center"); ax.set_yticklabels(DS_LABELS)
    ax.tick_params(labelsize=12)
    for i in range(n_ds):
        for j in range(n_ds):
            v = M[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white", fontsize=6)
    ax.grid(False)

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.015, pad=0.01)
cbar.set_label("AUROC", fontsize=16, rotation=270, labelpad=15)
cbar.ax.tick_params(labelsize=14)
fig.suptitle(f"Cross-task generalization (layer {FIG4_LAYER})", fontsize=16)
fig.savefig(f"fig4_generalization_heatmaps_{MODEL_NAME}.pdf", bbox_inches="tight")
plt.show()

---
## Figure 5: 2D projections of activations (own-probe subspace)

In [ ]:
# Figure 5: 3x3 scatter plots, each task projected onto its own probe + max-var residual
PROJ_LAYER = 25
PROJ_INSTR = "no-prompt"
MAX_PER_TASK = 2000
COLORS_TF  = {0: "#E8534A", 1: "#4393C3"}

def load_activations_matrix(ds, instruction, layer, max_n=None):
    pos, neg = load_activations(ds, instruction)
    if max_n:
        pos, neg = pos[:max_n], neg[:max_n]
    X, y = [], []
    for rec in pos:
        v = last_token_vec(rec, layer)
        if v is not None: 
            X.append(v)
            y.append(1)
    for rec in neg:
        v = last_token_vec(rec, layer)
        if v is not None: 
            X.append(v)
            y.append(0)
    return (np.stack(X).astype(np.float32), np.array(y)) if X else (np.empty((0,0)), np.empty(0))

grid = [ARITH_DS, FACT_DS[:3], FACT_DS[3:]]

fig, axes = plt.subplots(3, 3, figsize=(14, 10), dpi=300)
for r in range(3):
    for c in range(3):
        ax = axes[r, c]
        ds = grid[r][c]
        W, MU = load_probe(ds, PROJ_INSTR)
        w = W[PROJ_LAYER].astype(np.float32)
        mu = MU.get(PROJ_LAYER, np.zeros_like(w)).astype(np.float32)
        u = w / (np.linalg.norm(w) + 1e-8)

        X, y = load_activations_matrix(ds, PROJ_INSTR, PROJ_LAYER, MAX_PER_TASK)
        Xc = X - mu
        a = Xc @ u
        resid = Xc - np.outer(a, u)
        _, _, Vt = np.linalg.svd(resid, full_matrices=False)
        b = resid @ (Vt[0] / (np.linalg.norm(Vt[0]) + 1e-8))

        for lab in [0, 1]:
            m = y == lab
            ax.scatter(a[m], b[m], s=12, c=COLORS_TF[lab], linewidths=0, rasterized=True)
        ax.set_title(LABEL[ds], fontsize=22)
        ax.grid(alpha=0.25); sns.despine(ax=ax)

axes[1, 0].set_ylabel("Orthogonal direction of max. variance", fontsize=22)
axes[2, 1].set_xlabel("Truth direction", fontsize=22)
fig.legend(
    [Line2D([],[],marker="o",color="w",markerfacecolor=COLORS_TF[0],ms=8),
     Line2D([],[],marker="o",color="w",markerfacecolor=COLORS_TF[1],ms=8)],
    ["False", "True"], loc="lower center", ncol=2, frameon=False,
    bbox_to_anchor=(0.5, -0.02), fontsize=14)
fig.tight_layout()
fig.savefig(f"fig5_2d_projections_{MODEL_NAME}.pdf", bbox_inches="tight")
plt.show()